In [1]:
from huggingface_hub import hf_hub_download

# Download the model file
model_path = hf_hub_download(repo_id="LaucoTec/TC3007C_Inpainting", filename="models/InpaintingAE/inpainting.keras")

# Print the path to confirm
print(f"Model downloaded to: {model_path}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


models/InpaintingAE/inpainting.keras:   0%|          | 0.00/70.0M [00:00<?, ?B/s]

Model downloaded to: /root/.cache/huggingface/hub/models--LaucoTec--TC3007C_Inpainting/snapshots/8e7d6b5c3992f7d0ffbeb746399d6d7d4eb409e1/models/InpaintingAE/inpainting.keras


In [2]:
from tensorflow.keras.models import load_model

# Load the model from the downloaded path
model = load_model(model_path, compile=False)

In [3]:
import numpy as np
import cv2

def predict_inpainting(data):
    # Extract the original image and the mask
    image = data['image']
    mask = data['mask']

    # Fix: Convert RGBA to RGB if necessary
    if len(image.shape) == 3 and image.shape[2] == 4:
        image = cv2.cvtColor(image, cv2.COLOR_RGBA2RGB)

    # Process the mask to ensure it's a single channel binary mask
    if len(mask.shape) == 3:
        # If mask is RGB/RGBA, convert to Gray
        mask = cv2.cvtColor(mask, cv2.COLOR_RGB2GRAY)

    # Threshold the mask so drawn areas are clearly non-zero
    _, mask = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)

    # Get the model's expected input dimensions
    try:
        target_h = model.input_shape[1]
        target_w = model.input_shape[2]
    except:
        target_h, target_w = None, None

    # Resize logic
    if target_h is not None and target_w is not None:
        processed_image = cv2.resize(image, (target_w, target_h))
        processed_mask = cv2.resize(mask, (target_w, target_h), interpolation=cv2.INTER_NEAREST)
    else:
        # Fallback
        h, w, _ = image.shape
        new_h = ((h + 7) // 8) * 8
        new_w = ((w + 7) // 8) * 8
        processed_image = cv2.resize(image, (new_w, new_h))
        processed_mask = cv2.resize(mask, (new_w, new_h), interpolation=cv2.INTER_NEAREST)

    # Apply the mask: set pixels to 0 where the mask is present
    masked_image = processed_image.copy()
    masked_image[processed_mask > 0] = 0

    # Normalize to [0, 1] and add batch dimension
    input_data = masked_image.astype('float32') / 255.0
    input_data = np.expand_dims(input_data, axis=0)

    # Run model inference
    try:
        # verbose=0 suppresses the progress bar
        prediction = model.predict(input_data, verbose=0)

        # Post-process the output
        reconstructed = np.squeeze(prediction, axis=0)
        reconstructed = np.clip(reconstructed, 0.0, 1.0)
        reconstructed = (reconstructed * 255).astype(np.uint8)
        return reconstructed
    except Exception as e:
        print(f"Error during inference: {e}")
        raise e

In [4]:
import gradio as gr
# Wrapper to handle the input from Gradio's ImageEditor
def inpainting_wrapper(data):
    if not data or not data["background"] is not None:
        return None

    # Extract image and mask from the dictionary returned by ImageEditor
    image = data["background"]

    # The drawing is in the layers; usually layer 0
    # If no drawing, create an empty mask
    if data["layers"]:
        mask_layer = data["layers"][0]
    else:
        mask_layer = np.zeros_like(image)

    # Use the alpha channel of the drawing layer as the mask if it exists
    if len(mask_layer.shape) == 3 and mask_layer.shape[-1] == 4:
        mask = mask_layer[:, :, 3]
    else:
        mask = mask_layer

    return predict_inpainting({'image': image, 'mask': mask})

# Initialize the Gradio interface
interface = gr.Interface(
    fn=inpainting_wrapper,
    inputs=gr.ImageEditor(type="numpy", label="Upload Image and Draw Mask"),
    outputs=gr.Image(label="Reconstructed Image"),
    title="Image Inpainting App",
    description="Upload an image and draw a mask to reconstruct the hidden parts."
)

# Launch the application cleanly without debug logs
interface.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b72535ff40ac69a41d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
